# 01_ETL: Transaction Risk Data Cleaning

This notebook loads the raw SAML-D transaction dataset, performs exploratory inspection, cleans and transforms the data, and exports a cleaned dataset for further analysis.


## Setup and Load Data

Import required libraries and load the raw transaction dataset using relative paths. We use `pathlib.Path` to avoid hardcoded absolute paths.

In [ ]:

from pathlib import Path
import pandas as pd

# Define paths
DATA_RAW_DIR = Path('data/raw')
DATA_PROCESSED_DIR = Path('data/processed')
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load raw data (adjust delimiter if needed)
raw_path = DATA_RAW_DIR / 'SAML-D_reduced.txt'
df_raw = pd.read_csv(raw_path)

# Inspect basic structure
print('Shape:', df_raw.shape)
print(df_raw.head())
print(df_raw.info())
print(df_raw.isnull().sum())


## Data Cleaning

Standardize column names, convert data types, handle missing values, and remove duplicates.

In [ ]:

# Clean column names

df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Example: convert date and time columns if present
if {'date','time'}.issubset(df.columns):
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')

# Convert numeric columns where appropriate
for col in df.select_dtypes(include=['object']).columns:
    # Attempt to convert to numeric where possible
    df[col] = pd.to_numeric(df[col], errors='ignore')

# Handle missing values (drop rows with missing critical fields)
critical_cols = ['amount','sender_account','receiver_account']
for col in critical_cols:
    if col in df.columns:
        df = df[df[col].notnull()]

# Fill remaining missing values with median or appropriate statistic
df = df.fillna(df.median(numeric_only=True))

# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Removed {before-after} duplicate rows')


## Feature Engineering (Basic Risk Fields)

Create simple risk-related features such as transaction amount bands and large transaction flag.

In [ ]:

# Create amount band
if 'amount' in df.columns:
    df['amount_band'] = pd.qcut(df['amount'], q=4, labels=['low','medium','high','very_high'])
    # Flag transactions above the 95th percentile as potentially risky
    threshold = df['amount'].quantile(0.95)
    df['large_transaction_flag'] = (df['amount'] > threshold).astype(int)

# Example risk ratio by account (if sender_account exists)
if 'sender_account' in df.columns:
    laundering_by_sender = df.groupby('sender_account')['is_laundering'].mean() if 'is_laundering' in df.columns else None
    # Map back to df as a feature if exists
    if laundering_by_sender is not None:
        df['sender_risk_ratio'] = df['sender_account'].map(laundering_by_sender)


## Save Cleaned Data

Save the cleaned and enriched dataset to the processed directory.

In [ ]:

# Save cleaned dataset
processed_file = DATA_PROCESSED_DIR / 'cleaned_transactions.csv'
df.to_csv(processed_file, index=False)
print('Cleaned data saved to', processed_file)
